In [6]:
#@title Import Library

!pip install -q kagglehub gdown scikit-image tqdm

import os
import json
import zipfile
import tarfile
import hashlib
import joblib
import cv2
import kagglehub
import numpy as np
import pandas as pd
import tensorflow as tf
from PIL import Image, UnidentifiedImageError
from tqdm.auto import tqdm
from skimage.feature import hog
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_class_weight

In [1]:
#@title Download Dataset

path_waste = kagglehub.dataset_download("techsash/waste-classification-data")
print("1. Waste Classification path:", path_waste)

TRASHNET_DIR = "/content/trashnet"
os.makedirs(TRASHNET_DIR, exist_ok=True)
if not os.path.exists(os.path.join(TRASHNET_DIR, "dataset-resized")):
    !wget -q https://github.com/garythung/trashnet/raw/master/data/dataset-resized.zip -O /content/trashnet.zip
    with zipfile.ZipFile("/content/trashnet.zip", "r") as zf:
        zf.extractall(TRASHNET_DIR)
print("2. TrashNet folders:", os.listdir(os.path.join(TRASHNET_DIR, "dataset-resized")))

GDRIVE_DIR = "/content/gdrive_dataset"
os.makedirs(GDRIVE_DIR, exist_ok=True)
GDRIVE_ARCHIVE = "/content/dataset_gdrive_archive"
!gdown --fuzzy "https://drive.google.com/file/d/1o101UBJGeeMPpI-DSY6oh-tLk9AHXMny/view?usp=sharing" -O {GDRIVE_ARCHIVE}

if zipfile.is_zipfile(GDRIVE_ARCHIVE):
    with zipfile.ZipFile(GDRIVE_ARCHIVE, "r") as zf:
        zf.extractall(GDRIVE_DIR)
elif tarfile.is_tarfile(GDRIVE_ARCHIVE):
    with tarfile.open(GDRIVE_ARCHIVE, "r:*") as tf_arc:
        tf_arc.extractall(GDRIVE_DIR)
print("3. Isi folder Google Drive:", os.listdir(GDRIVE_DIR))

Using Colab cache for faster access to the 'waste-classification-data' dataset.
1. Waste Classification path: /kaggle/input/waste-classification-data
2. TrashNet folders: ['paper', 'metal', 'trash', '.DS_Store', 'plastic', 'glass', 'cardboard']
Downloading...
From (original): https://drive.google.com/uc?id=1o101UBJGeeMPpI-DSY6oh-tLk9AHXMny
From (redirected): https://drive.google.com/uc?id=1o101UBJGeeMPpI-DSY6oh-tLk9AHXMny&confirm=t&uuid=8ce42472-dcd8-4e94-8645-ab65c5c50f9d
To: /content/dataset_gdrive_archive
100% 1.45G/1.45G [00:12<00:00, 117MB/s] 
3. Isi folder Google Drive: ['JPEGImages', 'SegmentationClass', 'readme.txt', 'DepthImages', 'ImageSets']


In [2]:
#@title Setup Direktori Output & Reproducibility

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

CLEAN_IMG_DIR = "/content/data/cleaned_images"
PROCESSED_DIR = "/content/data/processed"

os.makedirs(CLEAN_IMG_DIR, exist_ok=True)
os.makedirs(PROCESSED_DIR, exist_ok=True)

print("Direktori output siap:")
print("- Citra bersih :", CLEAN_IMG_DIR)
print("- Artefak data :", PROCESSED_DIR)

Direktori output siap:
- Citra bersih : /content/data/cleaned_images
- Artefak data : /content/data/processed


In [3]:
#@title Data Cleaning, Deduplikasi & Target Derivation

VALID_EXT = (".jpg", ".jpeg", ".png", ".bmp", ".webp")
records = []

# Kumpulkan Metadata & Terapkan Target Derivation (0: Non-Recyclable, 1: Recyclable)
# Waste Classification Data (Kaggle)
waste_target_dir = os.path.join(path_waste, "DATASET")
if not os.path.exists(waste_target_dir):
    waste_target_dir = path_waste

for split_name in ["TRAIN", "TEST"]:
    for folder in ["O", "R"]:
        class_dir = os.path.join(waste_target_dir, split_name, folder)
        if os.path.exists(class_dir):
            target = 1 if folder == "R" else 0
            for fname in sorted(os.listdir(class_dir)):
                if fname.lower().endswith(VALID_EXT):
                    records.append({
                        "raw_path": os.path.join(class_dir, fname),
                        "source": "Waste_Classification",
                        "raw_class": folder,
                        "target": target
                    })

# TrashNet (GitHub)
trashnet_root = os.path.join(TRASHNET_DIR, "dataset-resized")
for cls_name in sorted(os.listdir(trashnet_root)):
    cls_dir = os.path.join(trashnet_root, cls_name)
    if os.path.isdir(cls_dir):
        target = 0 if cls_name.lower() == "trash" else 1
        for fname in sorted(os.listdir(cls_dir)):
            if fname.lower().endswith(VALID_EXT):
                records.append({
                    "raw_path": os.path.join(cls_dir, fname),
                    "source": "TrashNet",
                    "raw_class": cls_name.lower(),
                    "target": target
                })

# MJU-Waste (Google Drive - Github) (Recyclable Only)
gdrive_jpeg_dir = None
for root, dirs, _ in os.walk(GDRIVE_DIR):
    if "JPEGImages" in dirs:
        gdrive_jpeg_dir = os.path.join(root, "JPEGImages")
        break

if gdrive_jpeg_dir and os.path.exists(gdrive_jpeg_dir):
    for fname in sorted(os.listdir(gdrive_jpeg_dir)):
        if fname.lower().endswith(VALID_EXT):
            records.append({
                "raw_path": os.path.join(gdrive_jpeg_dir, fname),
                "source": "MJU_Waste",
                "raw_class": "JPEGImages",
                "target": 1
            })

df_raw = pd.DataFrame(records)
print("=== RINCIAN CITRA MENTAH TERKUMPUL ===")
print(df_raw["source"].value_counts())
print(f"Total citra mentah awal : {len(df_raw):,} citra\n")

# Data Cleaning: Validasi File, Standarisasi RGB & Deduplikasi MD5
CONVERTED_DIR = "/content/data/converted_rgb"
os.makedirs(CONVERTED_DIR, exist_ok=True)

clean_rows = []
seen_md5 = set()
stats = {"corrupt": 0, "converted_rgb": 0, "duplicate": 0}

for idx, row in tqdm(df_raw.iterrows(), total=len(df_raw), desc="Cleaning & Deduplikasi"):
    raw_path = row["raw_path"]
    if not os.path.exists(raw_path):
        stats["corrupt"] += 1
        continue

    try:
        with Image.open(raw_path) as img:
            # Konversi mode non-RGB (Palette 'P', RGBA, Grayscale 'L') menjadi RGB 3-channel
            if img.mode != "RGB":
                stats["converted_rgb"] += 1
                rgb_img = img.convert("RGB")
                clean_fname = f"{row['source']}_{idx:06d}_rgb.jpg"
                final_path = os.path.join(CONVERTED_DIR, clean_fname)
                rgb_img.save(final_path, format="JPEG", quality=95)
                width, height = rgb_img.size
            else:
                rgb_img = img
                final_path = raw_path
                clean_fname = os.path.basename(raw_path)
                width, height = img.size

            # Hitung MD5 hash dari byte piksel untuk menyaring citra duplikat identik
            img_md5 = hashlib.md5(rgb_img.tobytes()).hexdigest()
            if img_md5 in seen_md5:
                stats["duplicate"] += 1
                continue
            seen_md5.add(img_md5)

            clean_rows.append({
                "filename": clean_fname,
                "filepath": final_path,
                "source": row["source"],
                "raw_class": row["raw_class"],
                "target": row["target"],
                "severity_label": "Recyclable" if row["target"] == 1 else "Non-Recyclable",
                "width": width,
                "height": height
            })
    except (UnidentifiedImageError, OSError, Exception):
        stats["corrupt"] += 1

df_img = pd.DataFrame(clean_rows)

print("\n=== RINGKASAN DATA CLEANING & TARGET DERIVATION ===")
print(f"Citra awal               : {len(df_raw):,}")
print(f"Citra rusak dihapus      : {stats['corrupt']:,}")
print(f"Citra duplikat dihapus   : {stats['duplicate']:,}")
print(f"Citra non-RGB dikonversi : {stats['converted_rgb']:,}")
print(f"Total citra bersih siap pakai: {len(df_img):,}")
print("\nDistribusi Target per Sumber Dataset:")
print(pd.crosstab(df_img["source"], df_img["severity_label"], margins=True))

=== RINCIAN CITRA MENTAH TERKUMPUL ===
source
Waste_Classification    25077
TrashNet                 2527
MJU_Waste                2475
Name: count, dtype: int64
Total citra mentah awal : 30,079 citra



Cleaning & Deduplikasi:   0%|          | 0/30079 [00:00<?, ?it/s]


=== RINGKASAN DATA CLEANING & TARGET DERIVATION ===
Citra awal               : 30,079
Citra rusak dihapus      : 0
Citra duplikat dihapus   : 329
Citra non-RGB dikonversi : 372
Total citra bersih siap pakai: 29,750

Distribusi Target per Sumber Dataset:
severity_label        Non-Recyclable  Recyclable    All
source                                                 
MJU_Waste                          0        2475   2475
TrashNet                         137        2387   2524
Waste_Classification           13651       11100  24751
All                            13788       15962  29750


In [5]:
#@title Stratified Splitting (Train / Valid / Test) & Simpan Metadata

# Stratifikasi gabungan (sumber dataset + target biner) untuk perataan di tiap split
strata = df_img["source"] + "_" + df_img["target"].astype(str)
if strata.value_counts().min() < 2:
    strata = df_img["target"]

# Split 70% Train, 30% Temp
train_df, temp_df = train_test_split(
    df_img,
    test_size=0.30,
    random_state=RANDOM_STATE,
    stratify=strata
)

# Split 30% Temp to 15% Valid & 15% Test
strata_temp = temp_df["source"] + "_" + temp_df["target"].astype(str)
if strata_temp.value_counts().min() < 2:
    strata_temp = temp_df["target"]

val_df, test_df = train_test_split(
    temp_df,
    test_size=0.50,
    random_state=RANDOM_STATE,
    stratify=strata_temp
)

train_df = train_df.reset_index(drop=True)
val_df   = val_df.reset_index(drop=True)
test_df  = test_df.reset_index(drop=True)

print("=== RINGKASAN STRATIFIED SPLITTING (70% : 15% : 15%) ===")
print(f"Train : {len(train_df):,} citra | Non-Recyclable (0): {(train_df['target']==0).sum():,} | Recyclable (1): {(train_df['target']==1).sum():,}")
print(f"Valid : {len(val_df):,} citra | Non-Recyclable (0): {(val_df['target']==0).sum():,} | Recyclable (1): {(val_df['target']==1).sum():,}")
print(f"Test  : {len(test_df):,} citra | Non-Recyclable (0): {(test_df['target']==0).sum():,} | Recyclable (1): {(test_df['target']==1).sum():,}")

# Simpan metadata CSV
train_df.to_csv(os.path.join(PROCESSED_DIR, "train_meta.csv"), index=False)
val_df.to_csv(os.path.join(PROCESSED_DIR, "val_meta.csv"), index=False)
test_df.to_csv(os.path.join(PROCESSED_DIR, "test_meta.csv"), index=False)
print("\nMetadata CSV berhasil disimpan di:", PROCESSED_DIR)

=== RINGKASAN STRATIFIED SPLITTING (70% : 15% : 15%) ===
Train : 20,825 citra | Non-Recyclable (0): 9,652 | Recyclable (1): 11,173
Valid : 4,462 citra | Non-Recyclable (0): 2,068 | Recyclable (1): 2,394
Test  : 4,463 citra | Non-Recyclable (0): 2,068 | Recyclable (1): 2,395

Metadata CSV berhasil disimpan di: /content/data/processed


In [7]:
#@title Feature Engineering: Ekstraksi Color Histogram + HOG

def extract_classical_features(df_subset, img_size=(224, 224), desc="Extracting"):
    feature_list = []
    for path in tqdm(df_subset["filepath"], desc=desc):
        img = cv2.imread(path)
        img_resized = cv2.resize(img, img_size)

        # Color Histogram RGB (32 bins x 3 channel = 96 fitur)
        color_hist = []
        for ch in range(3):
            hist = cv2.calcHist([img_resized], [ch], None, [32], [0, 256])
            cv2.normalize(hist, hist)
            color_hist.extend(hist.flatten())

        # HOG Features pada citra Grayscale (6.084 fitur)
        gray = cv2.cvtColor(img_resized, cv2.COLOR_BGR2GRAY)
        hog_features = hog(
            gray,
            orientations=9,
            pixels_per_cell=(16, 16),
            cells_per_block=(2, 2),
            block_norm="L2-Hys",
            feature_vector=True
        )

        combined_vector = np.hstack([color_hist, hog_features])
        feature_list.append(combined_vector)

    return np.array(feature_list, dtype=np.float32)

print("Mengekstrak fitur Color Histogram + HOG...")
X_train_raw = extract_classical_features(train_df, desc="Train Features")
X_val_raw   = extract_classical_features(val_df, desc="Valid Features")
X_test_raw  = extract_classical_features(test_df, desc="Test Features")

y_train = train_df["target"].values
y_val   = val_df["target"].values
y_test  = test_df["target"].values

print("\nDimensi fitur mentah:")
print("- X_train_raw:", X_train_raw.shape)
print("- X_val_raw  :", X_val_raw.shape)
print("- X_test_raw :", X_test_raw.shape)

Mengekstrak fitur Color Histogram + HOG...


Train Features:   0%|          | 0/20825 [00:00<?, ?it/s]

Valid Features:   0%|          | 0/4462 [00:00<?, ?it/s]

Test Features:   0%|          | 0/4463 [00:00<?, ?it/s]


Dimensi fitur mentah:
- X_train_raw: (20825, 6180)
- X_val_raw  : (4462, 6180)
- X_test_raw : (4463, 6180)


In [8]:
#@title Scaling Data, Data Balancing (Class Weight) & Simpan Artefak

# Fit StandardScaler HANYA pada data training (mencegah data leakage)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train_raw)
X_val   = scaler.transform(X_val_raw)
X_test  = scaler.transform(X_test_raw)

# Hitung Class Weights HANYA pada data training
classes = np.unique(y_train)
weights = compute_class_weight(class_weight="balanced", classes=classes, y=y_train)
class_weight_dict = {int(c): float(w) for c, w in zip(classes, weights)}

# Simpan dataset terproses, scaler & class_weights
np.savez_compressed(
    os.path.join(PROCESSED_DIR, "classical_features.npz"),
    X_train=X_train, y_train=y_train,
    X_val=X_val, y_val=y_val,
    X_test=X_test, y_test=y_test
)
joblib.dump(scaler, os.path.join(PROCESSED_DIR, "standard_scaler.joblib"))

with open(os.path.join(PROCESSED_DIR, "class_weights.json"), "w") as f:
    json.dump(class_weight_dict, f, indent=2)

print("=== RINGKASAN TRANSFORMASI & DATA BALANCING ===")
print(f"Class Weight 0 (Non-Recyclable): {class_weight_dict[0]:.4f}")
print(f"Class Weight 1 (Recyclable)    : {class_weight_dict[1]:.4f}")
print("\nSelesai! Seluruh artefak tersimpan di:", PROCESSED_DIR)
print("Daftar file:", sorted(os.listdir(PROCESSED_DIR)))

=== RINGKASAN TRANSFORMASI & DATA BALANCING ===
Class Weight 0 (Non-Recyclable): 1.0788
Class Weight 1 (Recyclable)    : 0.9319

Selesai! Seluruh artefak tersimpan di: /content/data/processed
Daftar file: ['class_weights.json', 'classical_features.npz', 'standard_scaler.joblib', 'test_meta.csv', 'train_meta.csv', 'val_meta.csv']


In [9]:
#@title Pipeline Transformasi & Augmentasi Citra

IMG_SIZE = (224, 224)
BATCH_SIZE = 32

# Layer augmentasi
data_augmentation = tf.keras.Sequential([
    tf.keras.layers.RandomFlip("horizontal_and_vertical", seed=RANDOM_STATE),
    tf.keras.layers.RandomRotation(0.2, seed=RANDOM_STATE),
    tf.keras.layers.RandomZoom(0.15, seed=RANDOM_STATE),
    tf.keras.layers.RandomContrast(0.15, seed=RANDOM_STATE)], name="train_augmentation")

def load_and_preprocess_image(filepath, label, is_training=False):
    img = tf.io.read_file(filepath)
    img = tf.io.decode_image(img, channels=3, expand_animations=False)
    img.set_shape([None, None, 3])
    img = tf.image.resize(img, IMG_SIZE)

    if is_training:
        img = data_augmentation(img, training=True)

    # Normalisasi piksel [-1, 1]
    img = tf.keras.applications.mobilenet_v2.preprocess_input(img)
    return img, label

def create_tf_dataset(df_subset, is_training=False):
    ds = tf.data.Dataset.from_tensor_slices((df_subset["filepath"].values, df_subset["target"].values))
    if is_training:
        ds = ds.shuffle(buffer_size=min(len(df_subset), 10000), seed=RANDOM_STATE)
    ds = ds.map(lambda x, y: load_and_preprocess_image(x, y, is_training), num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
    return ds

train_ds = create_tf_dataset(train_df, is_training=True)
val_ds   = create_tf_dataset(val_df, is_training=False)
test_ds  = create_tf_dataset(test_df, is_training=False)

# Verifikasi 1 batch pertama dari train_ds
for sample_imgs, sample_labels in train_ds.take(1):
    print("=== VERIFIKASI PIPELINE DEEP LEARNING ===")
    print(f"Batch shape train_ds : {sample_imgs.shape}")
    print(f"Batch labels shape   : {sample_labels.shape}")
    print(f"Rentang nilai piksel : [{sample_imgs.numpy().min():.2f}, {sample_imgs.numpy().max():.2f}]")

=== VERIFIKASI PIPELINE DEEP LEARNING ===
Batch shape train_ds : (32, 224, 224, 3)
Batch labels shape   : (32,)
Rentang nilai piksel : [-1.00, 1.00]
